# Install Libraries

In [1]:
!pip install ultralytics
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 24.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics] [ultralytics]


# Imports

In [1]:
import cv2
import numpy as np
import os
import random
import shutil
import matplotlib.pyplot as plt
from roboflow import Roboflow

# Get Dataset Keypoints

In [2]:
roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")


from roboflow import Roboflow

rf = Roboflow(api_key=roboflow_api_key)

workspace = rf.workspace("valentin-weyer-xasiu")
project = workspace.project("keypointv333-uwois-xprdi")
version = project.version(1)
dataset = version.download("yolov8")

workspace = rf.workspace("valentin-weyer-xasiu")


loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...


In [8]:
#!yolo task=pose mode=train model=yolo11x-pose.pt data={dataset.location}/data.yaml batch=32 epochs=500 imgsz=640

#!yolo task=pose mode=train model=yolov8x-pose.pt data={dataset.location}/data.yaml batch=64 epochs=500 imgsz=640 mosaic=0.0

!yolo task=pose mode=train model=yolov8x-pose.pt data={dataset.location}/data.yaml epochs=300 imgsz=960 batch=16 mosaic=0.0 auto_augment=None erasing=0.0 lr0=0.003

New https://pypi.org/project/ultralytics/8.3.239 available 😃 Update with 'pip install -U ultralytics'
/home/valentinweyer/miniforge3/envs/NewEnv/lib/python3.11/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(
Ultralytics 8.3.237 🚀 Python-3.11.14 torch-2.9.1+cu130 CUDA:0 (NVIDIA GB10, 122506MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/valentinweyer/projects/handball-computer-vision/notebooks/keypointv333-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.0, exist_ok=False, fliplr=0.5, 

In [9]:
workspace.deploy_model(
    model_type="yolov11x",  # Type of the model
    model_path="/home/valentinweyer/runs/pose/train",  # Path to model directory
    project_ids=["keypointv333-uwois-xprdi"],  # List of project IDs
    model_name="yolov11x-keypoint",  # Name for the model (must have at least 1 letter, and accept numbers and dashes)
    filename="weights/best.pt"  # Path to weights file (default)
)

View the status of your deployment for project keypointv333-uwois-xprdi at: https://app.roboflow.com/valentin-weyer-xasiu/keypointv333-uwois-xprdi/models


In [3]:
from ultralytics import YOLO

model = YOLO("/home/valentinweyer/runs/pose/train2/weights/best.pt")

model.predict(
    source="/home/valentinweyer/projects/handball-computer-vision/source/Aalborg_Veszpreem_mp4-0146.jpg",
    imgsz=640,
    conf=0.6,
    save=True
)


    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    



image 1/1 /home/valentinweyer/projects/handball-computer-vision/source/Aalborg_Veszpreem_mp4-0146.jpg: 384x640 1 Court, 26.2ms
Speed: 1.3ms preprocess, 26.2ms inference, 11.2ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /home/valentinweyer/runs/pose/predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: ultralytics.engine.results.Keypoints object
 masks: None
 names: {0: 'Court'}
 obb: None
 orig_img: array([[[ 30,  36,  55],
         [ 29,  35,  54],
         [ 28,  34,  53],
         ...,
         [ 35,  36,  96],
         [ 30,  33,  94],
         [ 26,  29,  90]],
 
        [[ 22,  28,  47],
         [ 21,  27,  46],
         [ 21,  27,  46],
         ...,
         [ 27,  28,  89],
         [ 24,  27,  88],
         [ 23,  26,  87]],
 
        [[ 20,  27,  44],
         [ 20,  27,  44],
         [ 20,  27,  44],
         ...,
         [ 25,  25,  89],
         [ 23,  25,  89],
         [ 23,  25,  89]],
 
        ...,
 
        [[ 87,  85,  77],
         [118, 116, 108],
         [118, 115, 110],
         ...,
         [208, 151, 126],
         [220, 164, 139],
         [105,  51,  26]],
 
        [[ 87,  85,  77],
         [121, 119, 111],
         [121, 118, 

In [10]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
from pathlib import Path

W = "/home/valentinweyer/runs/pose/train2/weights/best.pt"
IMG = "/home/valentinweyer/projects/handball-computer-vision/source/Aalborg_Veszpreem_mp4-0146.jpg"
OUT = "/home/valentinweyer/runs/pose/predict/overlay_lines.png"

# 1) Define which keypoints should be connected by lines (index pairs)
# Example placeholder — REPLACE with your real court connectivity.
EDGES = [
    # (0, 1), (1, 2), (2, 3), (3, 0),  # outer rectangle example
    # (4, 5), (5, 6), ...
]

CONF_KP = 0.25   # only draw lines if both endpoints have kp conf >= this
CONF_BOX = 0.25  # box conf for inference

model = YOLO(W)
r = model.predict(source=IMG, imgsz=640, conf=CONF_BOX, iou=0.6, save=False, verbose=False)[0]

if len(r.boxes) == 0:
    raise RuntimeError("No detections.")

# pick the largest box (courts are large; filters player-false-positives hard)
boxes = r.boxes.xyxy
areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
idx = int(torch.argmax(areas))

# get image + keypoints
img = r.orig_img.copy()  # BGR
xy = r.keypoints.xy[idx].cpu().numpy()      # (37, 2)
kc = r.keypoints.conf[idx].cpu().numpy()    # (37,)

# 2) Draw predicted keypoints as circles
for i, (x, y) in enumerate(xy):
    if kc[i] >= CONF_KP:
        cv2.circle(img, (int(x), int(y)), 4, (0, 255, 0), -1)
        cv2.putText(img, str(i), (int(x) + 6, int(y) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

# 3) Draw lines between keypoint pairs
for a, b in EDGES:
    if kc[a] >= CONF_KP and kc[b] >= CONF_KP:
        ax, ay = xy[a]
        bx, by = xy[b]
        cv2.line(img, (int(ax), int(ay)), (int(bx), int(by)), (255, 0, 0), 2)

Path(OUT).parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(OUT, img)
print("Saved:", OUT)

Saved: /home/valentinweyer/runs/pose/predict/overlay_lines.png
